# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

## Baseline Rule

A page should be prioritized for content refresh if:

- it has not been updated for a long time,
- it still receives meaningful impressions,
- its average Google position indicates ranking improvement opportunity.

These pages still have visibility but may have outdated content, making them good candidates for content refresh.

### Reason Codes

| Code | Meaning |
|---|---|
| STALE_VISIBLE_POSITION_OPPORTUNITY | Page is old, still receives impressions, and has ranking improvement opportunity |
| VISIBLE_POSITION_OPPORTUNITY | Page has visibility and ranking opportunity but is not stale |
| STALE_CONTENT | Page is old but does not have enough visibility signals |
| NO_SIGNAL | Page does not match baseline conditions |

### Action Label

REFRESH_CONTENT

## 2. Build the ranked queue (writes the CSV)

### Baseline Scoring Logic

The baseline score is a simple additive rule:

- +1 if content is stale
- +1 if page has visibility
- +1 if ranking opportunity exists

Maximum score = 3.
Higher scores indicate stronger refresh candidates.

In [50]:
# Import required libraries

import pandas as pd
import numpy as np
from pathlib import Path

In [51]:
# Load the FlyRank content refresh dataset

DATA_PATH = Path("../../data/raw/content_refresh_anonymized.csv")

df = pd.read_csv(DATA_PATH)

df.shape

(30000, 44)

In [52]:
# View available columns

df.columns.tolist()

['content_id',
 'client_id',
 'search_volume',
 'competition',
 'competition_level',
 'cpc',
 'content_type',
 'main_intent',
 'word_count',
 'char_count',
 'provider_used',
 'model_used',
 'impressions_90d',
 'clicks_90d',
 'pageviews_90d',
 'sessions_90d',
 'users_90d',
 'engaged_sessions_90d',
 'ai_sessions_90d',
 'scroll_events_90d',
 'days_with_impressions',
 'days_with_sessions',
 'impressions_last_30d',
 'clicks_last_30d',
 'sessions_last_30d',
 'impressions_prev_30d',
 'clicks_prev_30d',
 'sessions_prev_30d',
 'content_age_days',
 'age_tier',
 'age_tier_order',
 'days_since_last_update',
 'freshness_tier',
 'word_count_tier',
 'char_count_tier',
 'ctr',
 'avg_position',
 'engagement_rate',
 'scroll_rate',
 'ai_traffic_pct',
 'impression_tier',
 'position_tier',
 'trend_direction',
 'trend_pct']

In [53]:
# Check columns required for baseline scoring

required_columns = [
    "days_since_last_update",
    "impressions_90d",
    "avg_position"
]

df[required_columns].head()

,days_since_last_update,impressions_90d,avg_position
0,20,3803,10.6
1,25,15320,20.3
2,20,12581,36.5
3,22,11751,6.2
4,14,19140,44.0


In [54]:
# Create stale signal

df["stale"] = (
    df["days_since_last_update"] >= 180
).astype(int)


df["stale"].value_counts()

stale
0    29826
1      174
Name: count, dtype: int64

In [55]:
# Create visibility signal

df["visible"] = (
    df["impressions_90d"] >= 500
).astype(int)


df["visible"].value_counts()

visible
1    16726
0    13274
Name: count, dtype: int64

In [56]:
# Create ranking opportunity signal

df["position_opportunity"] = (
    df["avg_position"] >= 10
).astype(int)


df["position_opportunity"].value_counts()

position_opportunity
1    15962
0    14038
Name: count, dtype: int64

In [57]:
# Calculate baseline score

df["baseline_score"] = (
    df["stale"]
    + df["visible"]
    + df["position_opportunity"]
)


df["baseline_score"].value_counts().sort_index()

baseline_score
0     6450
1    14252
2     9284
3       14
Name: count, dtype: int64

In [58]:
# Assign human-readable reason codes

def assign_reason(row):

    if (
        row["stale"]
        and row["visible"]
        and row["position_opportunity"]
    ):
        return "STALE_VISIBLE_POSITION_OPPORTUNITY"

    elif (
        row["visible"]
        and row["position_opportunity"]
    ):
        return "VISIBLE_POSITION_OPPORTUNITY"

    elif row["stale"]:
        return "STALE_CONTENT"

    else:
        return "NO_SIGNAL"



df["reason_code"] = df.apply(
    assign_reason,
    axis=1
)


df["reason_code"].value_counts()

reason_code
NO_SIGNAL                             20594
VISIBLE_POSITION_OPPORTUNITY           9232
STALE_CONTENT                           160
STALE_VISIBLE_POSITION_OPPORTUNITY       14
Name: count, dtype: int64

In [59]:
# Create action label

df["action_label"] = np.where(
    df["baseline_score"] >= 2,
    "REFRESH_CONTENT",
    "NO_ACTION"
)


df["action_label"].value_counts()

action_label
NO_ACTION          20702
REFRESH_CONTENT     9298
Name: count, dtype: int64

In [60]:
# Rank pages by baseline score

baseline_queue = df.sort_values(
    by=[
        "baseline_score",
        "impressions_90d"
    ],
    ascending=[
        False,
        False
    ]
).copy()


baseline_queue.head(10)

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,impression_tier,position_tier,trend_direction,trend_pct,stale,visible,position_opportunity,baseline_score,reason_code,action_label
16751,content_cf56e2e2e282,client_7f2253d7e2,0.0,0.0,LOW,0.0,keyword article,informational,5125.0,33705.0,...,excellent,striking,down,-85.6,1,1,1,3,STALE_VISIBLE_POSITION_OPPORTUNITY,REFRESH_CONTENT
16514,content_7368877ea310,client_7f2253d7e2,0.0,0.0,LOW,0.0,keyword article,informational,2591.0,16498.0,...,excellent,page_3_5,down,-81.5,1,1,1,3,STALE_VISIBLE_POSITION_OPPORTUNITY,REFRESH_CONTENT
7021,content_1bfaa38ff26c,client_7f2253d7e2,0.0,0.0,LOW,0.0,keyword article,informational,3861.0,24672.0,...,good,page_3_5,down,-74.7,1,1,1,3,STALE_VISIBLE_POSITION_OPPORTUNITY,REFRESH_CONTENT
21268,content_0a91db491d14,client_7f2253d7e2,0.0,0.0,LOW,0.0,keyword article,informational,3478.0,21948.0,...,good,striking,down,-51.8,1,1,1,3,STALE_VISIBLE_POSITION_OPPORTUNITY,REFRESH_CONTENT
11489,content_5feee3994adb,client_7f2253d7e2,0.0,0.0,LOW,0.0,keyword article,transactional,3590.0,22780.0,...,good,page_3_5,down,-89.1,1,1,1,3,STALE_VISIBLE_POSITION_OPPORTUNITY,REFRESH_CONTENT
12045,content_c2d929d83eaa,client_7f2253d7e2,0.0,0.0,LOW,0.0,keyword article,informational,4758.0,30070.0,...,good,striking,down,-62.8,1,1,1,3,STALE_VISIBLE_POSITION_OPPORTUNITY,REFRESH_CONTENT
698,content_b16bd7307b39,client_7f2253d7e2,0.0,0.0,LOW,0.0,keyword article,informational,4329.0,27844.0,...,good,page_3_5,down,-69.7,1,1,1,3,STALE_VISIBLE_POSITION_OPPORTUNITY,REFRESH_CONTENT
5327,content_fe16a55cd13d,client_7f2253d7e2,0.0,0.0,LOW,0.0,keyword article,informational,3388.0,21742.0,...,good,striking,down,-52.2,1,1,1,3,STALE_VISIBLE_POSITION_OPPORTUNITY,REFRESH_CONTENT
26810,content_ecb6215e79fd,client_7f2253d7e2,0.0,0.0,LOW,0.0,keyword article,informational,4486.0,29333.0,...,good,page_3_5,down,-74.4,1,1,1,3,STALE_VISIBLE_POSITION_OPPORTUNITY,REFRESH_CONTENT
20837,content_928af3e22c80,client_7f2253d7e2,0.0,0.0,LOW,0.0,keyword article,informational,3118.0,20396.0,...,moderate,striking,down,-45.7,1,1,1,3,STALE_VISIBLE_POSITION_OPPORTUNITY,REFRESH_CONTENT


In [61]:
# Save baseline output CSV

OUTPUT_PATH = Path(
    "../../work/outputs/baseline_action_score.csv"
)


OUTPUT_PATH.parent.mkdir(
    parents=True,
    exist_ok=True
)


baseline_queue.to_csv(
    OUTPUT_PATH,
    index=False
)


print(f"Saved: {OUTPUT_PATH}")

Saved: ..\..\work\outputs\baseline_action_score.csv


In [62]:
# Read saved file and verify

baseline_check = pd.read_csv(
    OUTPUT_PATH
)


baseline_check.head(10)

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,impression_tier,position_tier,trend_direction,trend_pct,stale,visible,position_opportunity,baseline_score,reason_code,action_label
0,content_cf56e2e2e282,client_7f2253d7e2,0.0,0.0,LOW,0.0,keyword article,informational,5125.0,33705.0,...,excellent,striking,down,-85.6,1,1,1,3,STALE_VISIBLE_POSITION_OPPORTUNITY,REFRESH_CONTENT
1,content_7368877ea310,client_7f2253d7e2,0.0,0.0,LOW,0.0,keyword article,informational,2591.0,16498.0,...,excellent,page_3_5,down,-81.5,1,1,1,3,STALE_VISIBLE_POSITION_OPPORTUNITY,REFRESH_CONTENT
2,content_1bfaa38ff26c,client_7f2253d7e2,0.0,0.0,LOW,0.0,keyword article,informational,3861.0,24672.0,...,good,page_3_5,down,-74.7,1,1,1,3,STALE_VISIBLE_POSITION_OPPORTUNITY,REFRESH_CONTENT
3,content_0a91db491d14,client_7f2253d7e2,0.0,0.0,LOW,0.0,keyword article,informational,3478.0,21948.0,...,good,striking,down,-51.8,1,1,1,3,STALE_VISIBLE_POSITION_OPPORTUNITY,REFRESH_CONTENT
4,content_5feee3994adb,client_7f2253d7e2,0.0,0.0,LOW,0.0,keyword article,transactional,3590.0,22780.0,...,good,page_3_5,down,-89.1,1,1,1,3,STALE_VISIBLE_POSITION_OPPORTUNITY,REFRESH_CONTENT
5,content_c2d929d83eaa,client_7f2253d7e2,0.0,0.0,LOW,0.0,keyword article,informational,4758.0,30070.0,...,good,striking,down,-62.8,1,1,1,3,STALE_VISIBLE_POSITION_OPPORTUNITY,REFRESH_CONTENT
6,content_b16bd7307b39,client_7f2253d7e2,0.0,0.0,LOW,0.0,keyword article,informational,4329.0,27844.0,...,good,page_3_5,down,-69.7,1,1,1,3,STALE_VISIBLE_POSITION_OPPORTUNITY,REFRESH_CONTENT
7,content_fe16a55cd13d,client_7f2253d7e2,0.0,0.0,LOW,0.0,keyword article,informational,3388.0,21742.0,...,good,striking,down,-52.2,1,1,1,3,STALE_VISIBLE_POSITION_OPPORTUNITY,REFRESH_CONTENT
8,content_ecb6215e79fd,client_7f2253d7e2,0.0,0.0,LOW,0.0,keyword article,informational,4486.0,29333.0,...,good,page_3_5,down,-74.4,1,1,1,3,STALE_VISIBLE_POSITION_OPPORTUNITY,REFRESH_CONTENT
9,content_928af3e22c80,client_7f2253d7e2,0.0,0.0,LOW,0.0,keyword article,informational,3118.0,20396.0,...,moderate,striking,down,-45.7,1,1,1,3,STALE_VISIBLE_POSITION_OPPORTUNITY,REFRESH_CONTENT


## 3. Top-20 Review

The baseline queue prioritizes pages that show a combination of:
- content age,
- existing visibility,
- ranking improvement opportunity.

The review below checks whether each recommendation is reasonable and what could make it wrong.

In [63]:
# Select top 20 pages from baseline queue for manual review

top20 = baseline_queue.head(20)[
    [
        "content_id",
        "baseline_score",
        "reason_code",
        "action_label",
        "impressions_90d",
        "days_since_last_update",
        "avg_position"
    ]
]


top20

,content_id,baseline_score,reason_code,action_label,impressions_90d,days_since_last_update,avg_position
16751,content_cf56e2e2e282,3,STALE_VISIBLE_POSITION_OPPORTUNITY,REFRESH_CONTENT,61678,194,19.7
16514,content_7368877ea310,3,STALE_VISIBLE_POSITION_OPPORTUNITY,REFRESH_CONTENT,59472,194,24.8
7021,content_1bfaa38ff26c,3,STALE_VISIBLE_POSITION_OPPORTUNITY,REFRESH_CONTENT,25715,194,22.2
21268,content_0a91db491d14,3,STALE_VISIBLE_POSITION_OPPORTUNITY,REFRESH_CONTENT,13299,193,10.5
11489,content_5feee3994adb,3,STALE_VISIBLE_POSITION_OPPORTUNITY,REFRESH_CONTENT,7812,194,39.0
12045,content_c2d929d83eaa,3,STALE_VISIBLE_POSITION_OPPORTUNITY,REFRESH_CONTENT,7558,193,17.9
698,content_b16bd7307b39,3,STALE_VISIBLE_POSITION_OPPORTUNITY,REFRESH_CONTENT,4590,194,31.0
5327,content_fe16a55cd13d,3,STALE_VISIBLE_POSITION_OPPORTUNITY,REFRESH_CONTENT,4556,194,16.4
26810,content_ecb6215e79fd,3,STALE_VISIBLE_POSITION_OPPORTUNITY,REFRESH_CONTENT,4429,194,25.3
20837,content_928af3e22c80,3,STALE_VISIBLE_POSITION_OPPORTUNITY,REFRESH_CONTENT,1697,193,15.8


## Top-20 Review Findings

The baseline queue prioritizes pages based on three transparent signals:

- Content age (days since last update)
- Existing visibility (impressions in the last 90 days)
- Ranking improvement opportunity (average position)

The top-ranked pages are candidates for content refresh because they show a combination of these signals.

The review below evaluates whether the recommendation is reasonable and what could make it incorrect.

| Rank | Action | Reason Code | Confidence Note | What would make it wrong |
|---|---|---|---|---|
| 1 | REFRESH_CONTENT | STALE_VISIBLE_POSITION_OPPORTUNITY | High confidence: page is old, visible, and has ranking opportunity | Wrong if impressions come from irrelevant searches |
| 2 | REFRESH_CONTENT | STALE_VISIBLE_POSITION_OPPORTUNITY | High confidence: content is stale but still receives traffic signals | Wrong if users already find the content useful |
| 3 | REFRESH_CONTENT | STALE_VISIBLE_POSITION_OPPORTUNITY | Medium-high confidence: page has visibility but ranking improvement opportunity | Wrong if competitors are causing ranking pressure |
| 4 | REFRESH_CONTENT | STALE_VISIBLE_POSITION_OPPORTUNITY | Medium confidence: refresh may improve relevance | Wrong if technical SEO is the actual problem |
| 5 | REFRESH_CONTENT | STALE_VISIBLE_POSITION_OPPORTUNITY | Medium confidence: baseline signals indicate review opportunity | Wrong if content quality is already sufficient |
| 6 | REFRESH_CONTENT | STALE_VISIBLE_POSITION_OPPORTUNITY | Lower confidence: visibility exists but impact may be limited | Wrong if business value is low |
| 7 | REFRESH_CONTENT | STALE_VISIBLE_POSITION_OPPORTUNITY | Medium confidence: page matches refresh criteria | Wrong if search intent changed |
| 8 | REFRESH_CONTENT | STALE_VISIBLE_POSITION_OPPORTUNITY | Medium confidence: multiple signals are present | Wrong if another optimization is required |
| 9 | REFRESH_CONTENT | STALE_VISIBLE_POSITION_OPPORTUNITY | High confidence: strong visibility and stale signal | Wrong if impressions are not valuable |
| 10 | REFRESH_CONTENT | STALE_VISIBLE_POSITION_OPPORTUNITY | Medium confidence: page has improvement opportunity | Wrong if updates do not affect ranking |
| 11 | REFRESH_CONTENT | STALE_VISIBLE_POSITION_OPPORTUNITY | Medium confidence: page is old with enough visibility signals | Wrong if content refresh does not address the real issue |
| 12 | REFRESH_CONTENT | STALE_VISIBLE_POSITION_OPPORTUNITY | Medium confidence: ranking opportunity exists with stale content | Wrong if ranking changes are temporary |
| 13 | REFRESH_CONTENT | STALE_VISIBLE_POSITION_OPPORTUNITY | Medium confidence: page meets multiple baseline conditions | Wrong if user engagement is already strong |
| 14 | REFRESH_CONTENT | STALE_VISIBLE_POSITION_OPPORTUNITY | Medium confidence: refresh could improve search relevance | Wrong if the page needs technical improvements instead |
| 15 | REFRESH_CONTENT | VISIBLE_POSITION_OPPORTUNITY | Medium confidence: page has visibility and ranking opportunity but is not stale | Wrong if freshness is not the limiting factor |
| 16 | REFRESH_CONTENT | VISIBLE_POSITION_OPPORTUNITY | Medium confidence: page may benefit from optimization | Wrong if search demand is too low |
| 17 | REFRESH_CONTENT | VISIBLE_POSITION_OPPORTUNITY | Lower confidence: page has ranking opportunity without strong freshness signal | Wrong if refresh effort has limited return |
| 18 | REFRESH_CONTENT | VISIBLE_POSITION_OPPORTUNITY | Medium confidence: baseline detected possible improvement area | Wrong if content relevance is already high |
| 19 | REFRESH_CONTENT | VISIBLE_POSITION_OPPORTUNITY | Medium confidence: selected by transparent scoring rules | Wrong if manual review disagrees |
| 20 | REFRESH_CONTENT | VISIBLE_POSITION_OPPORTUNITY | Medium confidence: page shows measurable improvement opportunity | Wrong if other ranking factors dominate |

In [64]:
# Confirm top20 review contains required information

top20.columns.tolist()

['content_id',
 'baseline_score',
 'reason_code',
 'action_label',
 'impressions_90d',
 'days_since_last_update',
 'avg_position']

## 4. Weak Picks + Leakage Check


## Weak Picks

Some baseline recommendations may be weaker because the rule uses simple transparent signals.

| Pick | Why it may be weak | Additional check |
|---|---|---|
| Pages with low impressions but high position opportunity | Refreshing these pages may have limited impact because audience size is small | Check search demand and business value |
| Old pages with impressions but poor ranking | The issue may not be freshness; it could be content quality, search intent, or competition | Review queries and competitor pages |
| Pages selected only through thresholds | Simple rules cannot understand content quality or user satisfaction | Future ML models can combine more signals |

## Leakage Check

I verified that the baseline rule only uses information available at the decision time.

### Signals Used

- days_since_last_update
- impressions_90d
- avg_position


### Signals Not Used

- trend_direction
- trend_pct
- is_declining_label
- future performance metrics


The baseline is a transparent decision-support rule and will be used as a comparison point for future ML models.

## Self-check

- [x] Every section above is filled — markdown thinking and code are included
- [x] Notebook runs top to bottom without errors (Runtime → Run all)
- [x] No client names, URLs, or private queries included
- [x] Claims use careful words: observed, measured, directional, decision-support
- [x] Notebook committed under `work/notebooks/`